In [40]:
import pandas as pd 
import seaborn as sns 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

In [41]:
def load_data():
    return sns.load_dataset("titanic")

In [42]:
df = load_data()
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [43]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [44]:
X = df
Y = df
X,Y

(     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
 0           0       3    male  22.0      1      0   7.2500        S   Third   
 1           1       1  female  38.0      1      0  71.2833        C   First   
 2           1       3  female  26.0      0      0   7.9250        S   Third   
 3           1       1  female  35.0      1      0  53.1000        S   First   
 4           0       3    male  35.0      0      0   8.0500        S   Third   
 ..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
 886         0       2    male  27.0      0      0  13.0000        S  Second   
 887         1       1  female  19.0      0      0  30.0000        S   First   
 888         0       3  female   NaN      1      2  23.4500        S   Third   
 889         1       1    male  26.0      0      0  30.0000        C   First   
 890         0       3    male  32.0      0      0   7.7500        Q   Third   
 
        who  adult_male deck  embark_t

In [45]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y,test_size = 0.2)

In [46]:
class DataValidationError(Exception):
    pass

def validate_data(df):
    required = ["survived", "sex", "age", "fare"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise DataValidationError(f"Missing required columns: {missing}")

In [47]:
def encode_categoricals(df):
    df["sex"] = df["sex"].map({"male": 0, "female": 1})
    df["class"] = df["class"].map({"first": 1, "second": 2, "third": 3})
    df = pd.get_dummies(df, columns=["embarked"], drop_first=True, dtype=int)
    return df

In [48]:
def encode_categoricals(df):
    df = df.copy()

    df["sex"] = df["sex"].map({"male": 0, "female": 1})
    df["class"] = df["class"].map({"first": 1, "second": 2, "third": 3})

    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    df = pd.get_dummies(
        df,
        columns=categorical_cols,
        drop_first=True,
        dtype=int
    )

    for col in df.select_dtypes(include="bool").columns:
        df[col] = df[col].astype(int)

    return df

In [49]:
def clean_text(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip().str.lower()
    return df

In [50]:
def drop_useless(df):
    return df.drop(columns=["alive", "deck"], errors="ignore")

In [51]:
def handle_missing(df):
    df = df.copy()

    for col in df.select_dtypes(include="number").columns:
        df[col] = df[col].fillna(df[col].median())

    for col in df.select_dtypes(include=["object", "category"]).columns:
        df[col] = df[col].fillna(df[col].mode()[0])

    return df

In [52]:
def split_data(df):
    X = df.drop("survived", axis=1)
    y = df["survived"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

In [53]:
def scale_features(X_train, X_test):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled

In [54]:
def main():
    try:
        df = load_data()
        validate_data(df)
        df = clean_text(df)
        df = drop_useless(df)
        df = handle_missing(df)
        df = encode_categoricals(df)

        X_train, X_test, y_train, y_test = split_data(df)

        X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

        print("Pipeline completed successfully.")
        print("Train shape:", X_train_scaled.shape)
        print("Test shape:", X_test_scaled.shape)

        return X_train_scaled, X_test_scaled, y_train, y_test

    except DataValidationError as e:
        print("Data validation error:", e)
    except Exception as e:
        print("Error:", e)

In [55]:
X_train_scaled, X_test_scaled, y_train, y_test = main()

Pipeline completed successfully.
Train shape: (712, 15)
Test shape: (179, 15)


C:\Users\varsh\AppData\Local\Temp\ipykernel_11636\2723586832.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\varsh\AppData\Local\Temp\ipykernel_11636\542763610.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m